# Big Data Bottleneck Analysis dengan PySpark

Analisis ini memproses dataset `scan_events.csv` asli yang dihasilkan oleh skrip Python untuk mengidentifikasi hub *bottleneck* sesungguhnya berdasarkan *average dwell time* tertinggi.

In [6]:
# Install pyspark (Uncomment jika dijalankan di Google Colab)
!pip install pyspark

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("BottleneckAnalysis") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()


### 1. Load Dataset `scan_events`
Membaca file CSV hasil generate dan melakukan konversi tipe data timestamp. Baris dengan timestamp yang rusak akan otomatis menjadi `NULL`.

In [7]:
import os

# Tentukan path file untuk Google Colab atau environment lokal Anda
file_path = "scan_events.csv"  # Jika Anda langsung mengunggah ke root folder Colab

if not os.path.exists(file_path):
    # Mencoba path alternatif jika dijalankan di sub-folder
    alternative_path = "../../generate-big-data/scan_events.csv"
    if os.path.exists(alternative_path):
        file_path = alternative_path
    else:
        print(f"[Peringatan] File tidak ditemukan di '{file_path}' maupun '{alternative_path}'.")
        print("Silakan unggah file 'scan_events.csv' ke folder Colab Anda di sebelah kiri (ikon folder).")

try:
    df = spark.read.csv(file_path, header=True, inferSchema=False) \
    .withColumn("timestamp", F.try_to_timestamp("timestamp"))

    print(f"Total baris mentah: {df.count()}")
    df.printSchema()
    df.show(5)
except Exception as e:
    print(f"Gagal membaca file: {e}")

Total baris mentah: 9286
root
 |-- hub_id: string (nullable = true)
 |-- package_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+-------+----------+----------+-------------------+
| hub_id|package_id|event_type|          timestamp|
+-------+----------+----------+-------------------+
|HUB-009| PKG-00301|   ARRIVAL|2023-10-26 12:27:13|
|HUB-010| PKG-00792|   ARRIVAL|2023-10-23 15:53:57|
|HUB-002| PKG-00064| DEPARTURE|2023-10-30 01:40:15|
|HUB-006| PKG-00364|   ARRIVAL|2023-10-05 04:15:51|
|HUB-009| PKG-00745|   ARRIVAL|2023-10-25 15:02:12|
+-------+----------+----------+-------------------+
only showing top 5 rows


### 2. Data Quality Check
Membersihkan data dengan tahapan berikut:
1. Menghapus event selain `ARRIVAL` dan `DEPARTURE` (membuang noise `SORTING`).
2. Menghapus data duplikat.
3. Memasangkan Arrival & Departure, lalu mendeteksi yang *missing* atau timestamp terbalik.

In [8]:
# 1. Filter hanya ARRIVAL dan DEPARTURE
df_filtered = df.filter(F.col("event_type").isin("ARRIVAL", "DEPARTURE"))

# 2. Hapus Duplicate Scans
df_clean = df_filtered.dropDuplicates()

# Pisahkan tabel ARRIVAL dan DEPARTURE
df_arrival = df_clean.filter(F.col("event_type") == "ARRIVAL") \
    .withColumnRenamed("timestamp", "arrival_time") \
    .drop("event_type")

df_departure = df_clean.filter(F.col("event_type") == "DEPARTURE") \
    .withColumnRenamed("timestamp", "departure_time") \
    .drop("event_type")

# 3. Gabungkan berdasarkan package_id dan hub_id menggunakan FULL OUTER JOIN
df_joined = df_arrival.join(df_departure, on=["package_id", "hub_id"], how="full")

# Cache dataframe karena akan dihitung berulang kali untuk reporting data kotor
df_joined.cache()

# Laporan Data Quality
missing_arrival = df_joined.filter(F.col("arrival_time").isNull()).count()
missing_departure = df_joined.filter(F.col("departure_time").isNull()).count()
invalid_time = df_joined.filter(F.col("departure_time") < F.col("arrival_time")).count()
null_timestamp = df_clean.filter(F.col("timestamp").isNull()).count()

print("=== HASIL DATA QUALITY CHECK ===")
print(f"Missing ARRIVAL: {missing_arrival}")
print(f"Missing DEPARTURE: {missing_departure}")
print(f"DEPARTURE mendahului ARRIVAL: {invalid_time}")
print(f"Timestamp rusak/NULL (setelah dibersihkan): {null_timestamp}")

=== HASIL DATA QUALITY CHECK ===
Missing ARRIVAL: 105
Missing DEPARTURE: 145
DEPARTURE mendahului ARRIVAL: 58
Timestamp rusak/NULL (setelah dibersihkan): 58


### 3. Hitung Dwell Time
Mengambil hanya pasangan ARRIVAL -> DEPARTURE yang valid dan menghitung durasi *dwell time* dalam satuan jam.

In [9]:
# Ambil pasangan yang valid saja
df_valid = df_joined.filter(
    F.col("arrival_time").isNotNull() &
    F.col("departure_time").isNotNull() &
    (F.col("departure_time") >= F.col("arrival_time"))
)
print(f"Total Pasangan Bersih/Valid: {df_valid.count()}")

# Hitung selisih dalam detik, lalu konversi ke Jam
df_dwell = df_valid.withColumn(
    "dwell_time_hours",
    (F.unix_timestamp("departure_time") - F.unix_timestamp("arrival_time")) / 3600.0
)
df_dwell.show(5)

Total Pasangan Bersih/Valid: 4253
+----------+-------+-------------------+-------------------+------------------+
|package_id| hub_id|       arrival_time|     departure_time|  dwell_time_hours|
+----------+-------+-------------------+-------------------+------------------+
| PKG-00153|HUB-002|2023-10-30 11:23:52|2023-10-30 13:47:25|            2.3925|
| PKG-00388|HUB-007|2023-10-04 12:08:20|2023-10-04 20:45:53| 8.625833333333333|
| PKG-00516|HUB-004|2023-10-16 11:35:43|2023-10-16 14:59:41|3.3994444444444443|
| PKG-00670|HUB-011|2023-10-05 11:57:26|2023-10-05 19:22:40| 7.420555555555556|
| PKG-00916|HUB-009|2023-10-05 15:05:27|2023-10-05 17:45:16| 2.663611111111111|
+----------+-------+-------------------+-------------------+------------------+
only showing top 5 rows


### 4. Aggregate per Hub
Kelompokkan berdasarkan Hub dan urutkan dari rata-rata *dwell time* tertinggi.

In [10]:
df_agg = df_dwell.groupBy("hub_id").agg(
    F.count("package_id").alias("package_count"),
    F.round(F.avg("dwell_time_hours"), 2).alias("avg_dwell_hours"),
    F.round(F.percentile_approx("dwell_time_hours", 0.5), 2).alias("median_dwell_hours")
)

# Urutkan
df_top_hubs = df_agg.orderBy(F.col("avg_dwell_hours").desc())
df_top_hubs.show()

+-------+-------------+---------------+------------------+
| hub_id|package_count|avg_dwell_hours|median_dwell_hours|
+-------+-------------+---------------+------------------+
|HUB-010|          150|          13.45|              6.71|
|HUB-005|          477|          12.07|             10.98|
|HUB-011|          247|           9.77|              8.63|
|HUB-003|          501|           7.59|              6.91|
|HUB-007|          277|           6.93|              6.65|
|HUB-002|          492|            6.6|              6.18|
|HUB-006|          374|           5.96|              5.49|
|HUB-009|          223|           5.57|              5.29|
|HUB-001|          603|           5.31|              4.91|
|HUB-012|          220|           5.16|               5.0|
|HUB-004|          432|           4.73|               4.5|
|HUB-008|          257|           4.22|              4.07|
+-------+-------------+---------------+------------------+



### 5. Identifikasi Bottleneck & Simpan ke CSV
Gunakan data di atas untuk menjawab `Business Insight` (Bandingkan rata-rata, median, dan volume paket). Lalu export hasil akhirnya ke `output/top_bottleneck_hubs.csv`.

In [11]:
import os
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

# Export ke CSV
pandas_df = df_top_hubs.toPandas()
pandas_df.to_csv(f"{output_dir}/top_bottleneck_hubs.csv", index=False)
print("Berhasil disimpan di output/top_bottleneck_hubs.csv!")

Berhasil disimpan di output/top_bottleneck_hubs.csv!


In [12]:
import shutil

# Menyalin file agar muncul langsung di folder utama (/content/) Google Colab agar mudah diunduh
try:
    shutil.copy("../output/top_bottleneck_hubs.csv", "./top_bottleneck_hubs.csv")
    print("File berhasil disalin ke folder utama! Silakan segarkan panel berkas di kiri Anda dan langsung unduh 'top_bottleneck_hubs.csv'.")
except Exception as e:
    print(f"Gagal menyalin file: {e}")

File berhasil disalin ke folder utama! Silakan segarkan panel berkas di kiri Anda dan langsung unduh 'top_bottleneck_hubs.csv'.
